Importing libraries and loading the dataset

In [123]:
import numpy as np
import pandas as pd

attrition_df = pd.read_csv('hr_attrition_dataset.csv')

Drop duplicates and check data info

In [124]:
attrition_df = attrition_df.drop_duplicates()
attrition_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   employee_id                  15000 non-null  str    
 1   age                          15000 non-null  int64  
 2   gender                       15000 non-null  str    
 3   marital_status               15000 non-null  str    
 4   education                    15000 non-null  str    
 5   education_field              15000 non-null  str    
 6   num_companies_worked         15000 non-null  int64  
 7   department                   15000 non-null  str    
 8   job_role                     15000 non-null  str    
 9   job_level                    15000 non-null  int64  
 10  employment_type              15000 non-null  str    
 11  years_at_company             15000 non-null  int64  
 12  years_in_current_role        15000 non-null  int64  
 13  total_working_years        

Null value imputation for performance_rating based on Department, Job Role, Job Level, Employment Type, Manager Rating, Skills Growth Opportunities

In [125]:
rating_hierarchy = [
    ['department', 'job_role', 'job_level', 'employment_type', 'manager_rating', 'skills_growth_opportunities'],
    ['department', 'job_role', 'job_level', 'employment_type', 'manager_rating'],
]

for group_cols in rating_hierarchy:
    attrition_df['performance_rating'] = attrition_df['performance_rating'].fillna(
        attrition_df.groupby(group_cols)['performance_rating'].transform('mean')
    )

attrition_df['performance_rating'] = attrition_df['performance_rating'].fillna(attrition_df['manager_rating'])

Null value imputation for Monthly Income Based on Department, Job Role, Job Level, Employment Type

In [126]:
group_cols_monthly_income = ['department', 'job_role','job_level','employment_type']

attrition_df['monthly_income'] = attrition_df['monthly_income'].fillna(
    attrition_df.groupby(group_cols_monthly_income)['monthly_income'].transform('mean')
)

Null value imputation for job satisfaction based on Department, Job Role, Job Level, Employment Type, Work Life Balance

In [127]:

group_cols_job_satisfaction = ['department', 'job_role','job_level','employment_type','work_life_balance']
attrition_df['job_satisfaction'] = attrition_df['job_satisfaction'].fillna(
    attrition_df.groupby(group_cols_job_satisfaction)['job_satisfaction'].transform('mean')
)
attrition_df['job_satisfaction'] = attrition_df['job_satisfaction'].fillna(attrition_df['work_life_balance'])

Creation of age groups based on categories: Gen Z, Early Career, Mid-Career, Experienced, Late Career

In [128]:
bins = [22, 25, 35, 45, 55, 65]
labels = ['Gen Z', 'Early Career', 'Mid-Career', 'Experienced', 'Late Career']

attrition_df['age_group'] = pd.cut(attrition_df['age'], bins=bins, labels=labels)

Creation of Average Tenure per Company based on Total Working Years and Number of Companies Worked

In [129]:
attrition_df['avg_tenure_per_company'] = (
    attrition_df['total_working_years'] / attrition_df['num_companies_worked'].replace(0, 1)
)
attrition_df['avg_tenure_per_company'] = attrition_df['avg_tenure_per_company'].round(2)

Creation of Role Stagnation Ratio based on Years in Current Role and Years at Company

In [130]:
attrition_df['role_stagnation_ratio'] = (
    attrition_df['years_in_current_role'] / attrition_df['years_at_company'].replace(0, 1)
)

attrition_df['role_stagnation_ratio'] = attrition_df['role_stagnation_ratio'].round(2)

Creation of Promotion Velocity based on Years at Company and Promotions in Last 5 Years

In [131]:
attrition_df['promotion_velocity'] = (
    attrition_df['years_at_company'] / (attrition_df['promotions_last_5years'])
)
attrition_df['promotion_velocity'] = attrition_df['promotion_velocity'].replace([np.inf, np.nan], 0)

Creation of Total Overtime Hours based on Overtime Hours per Week and Years in Current Role

In [132]:
attrition_df['total_overtime_hours'] = (
    attrition_df['overtime_hours_per_week'] * attrition_df['years_in_current_role'] * 52
)

Creation of Burnout Risk Score based on Overtime Hours per week, Overtime Paid Eligible

In [133]:
high_ot_threshold = 10

attrition_df['burnout_risk_flag'] = (
    (attrition_df['overtime_hours_per_week'] > high_ot_threshold) & 
    (attrition_df['overtime_paid_eligible'] == 'No')
)